# Notebook 1 — Full Multilingual Pipeline (EN + ES)
GPT‑4o Vision OCR → Embeddings → Auto‑Label → ML → Azure Endpoint → GPT‑4o Reasoning

In [ ]:
!pip install openai pandas tqdm pyarrow scikit-learn joblib requests

In [ ]:
from openai import OpenAI
import pandas as pd, numpy as np, time, os, joblib, requests
from tqdm import tqdm
client = OpenAI(api_key='YOUR_OPENAI_KEY')

## Step 1 — Load EN + ES Datasets

In [ ]:
df_en = pd.read_parquet('ENGLISH_SAS_URL')
df_es = pd.read_parquet('SPANISH_SAS_URL')
df_en['language']='en'
df_es['language']='es'
df = pd.concat([df_en, df_es]).reset_index(drop=True)
df.head()

## Step 2 — GPT‑4o Vision OCR

In [ ]:
def gpt4o_vision_ocr(b64):
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{ 'role':'user','content':[
            {'type':'text','text':'Extract ONLY the text from this image.'},
            {'type':'image_url','image_url':{'url':f'data:image/png;base64,{b64}'}}]}])
    return r.choices[0].message.content

In [ ]:
df['ocr_text']=None
COST=0.005; MAX=8.0; spent=0; count=0
for i in tqdm(range(len(df))):
    if df.loc[i,'ocr_text'] not in [None,'']: continue
    if spent + COST > MAX: break
    try:
        txt = gpt4o_vision_ocr(df.loc[i,'image'])
        df.loc[i,'ocr_text']=txt
        spent+=COST; count+=1
    except: time.sleep(1)
df.to_pickle('ocr_en_es.pkl'); count

## Step 3 — Embeddings

In [ ]:
def embed_text(t):
    if t is None or len(t.strip())==0: return None
    r = client.embeddings.create(model='text-embedding-3-large', input=t)
    return r.data[0].embedding

In [ ]:
df['embedding']=[embed_text(t) for t in tqdm(df['ocr_text'])]
df.to_pickle('embeddings_en_es.pkl')

## Step 4 — Auto‑Label (A1 Financial Sections)

In [ ]:
CATS=['Risk','Financials','Management','Operations','Legal','Other']
def auto_label(t):
    if t is None or len(t.strip())==0: return 'Other'
    r=client.chat.completions.create(model='gpt-4o-mini',messages=[
        {'role':'system','content':'Classify into ONE: '+', '.join(CATS)},
        {'role':'user','content':t}])
    out=r.choices[0].message.content.strip()
    return out if out in CATS else 'Other'

In [ ]:
df['label']=[auto_label(t) for t in tqdm(df['ocr_text'])]
df.to_pickle('labels_en_es.pkl')
df['label'].value_counts()

## Step 5 — Train Logistic Regression Model

In [ ]:
df_clean = df[df['embedding'].apply(lambda x: isinstance(x,list) and len(x)==3072)].reset_index(drop=True)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_clean['label_id'] = le.fit_transform(df_clean['label'])
X = np.vstack(df_clean['embedding'])
y = df_clean['label_id']
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=2000,class_weight='balanced')
model.fit(X,y)
joblib.dump(model,'fusion_model.joblib')
joblib.dump(le,'label_encoder.joblib')

## Step 6 — Generate Azure ML score.py

In [ ]:
%%writefile score.py
import json, numpy as np, joblib, os
def init():
    global model, le
    path=os.environ['AZUREML_MODEL_DIR']
    model=joblib.load(os.path.join(path,'fusion_model.joblib'))
    le=joblib.load(os.path.join(path,'label_encoder.joblib'))
def run(raw):
    d=json.loads(raw)
    emb=np.array(d['embedding'])
    pred=model.predict([emb])[0]
    score=float(model.predict_proba([emb])[0].max())
    label=le.inverse_transform([pred])[0]
    return {'predicted_class':label, 'score':score}

## Step 7 — Final Unified Pipeline Output

In [ ]:
def process_page(idx, use_endpoint=False, endpoint_url=None, endpoint_key=None):
    out={}
    text=df.loc[idx,'ocr_text']
    emb=df.loc[idx,'embedding']
    out['ocr_text']=text
    out['embedding']=emb
    if use_endpoint and endpoint_url:
        headers={'Authorization':f'Bearer {endpoint_key}'}
        r=requests.post(endpoint_url,json={'embedding':emb},headers=headers)
        try: j=r.json(); out['predicted_class']=j.get('predicted_class'); out['score']=j.get('score')
        except: out['predicted_class']=None; out['score']=None
    else: out['predicted_class']=None; out['score']=None
    resp=client.chat.completions.create(model='gpt-4o',messages=[
        {'role':'system','content':'You are a financial analyst.'},
        {'role':'user','content': f'''Summarise and explain this page:
{text}'''}])
    summary=resp.choices[0].message.content
    out['gpt4_summary']=summary
    out['gpt4_explanation']=summary
    return out

In [ ]:
process_page(0)